# Evaluation: Zero-Shot Baseline & SFT Inference

Evaluate function-calling performance across model variants:
1. Load Qwen2.5-3B-Instruct (zero-shot baseline)
2. Load SFT-trained LoRA adapters (V2 and V3)
3. Run inference on sample examples from the formatted dataset
4. Compare predicted tool calls against ground truth

**Runtime:** T4 GPU sufficient for inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/info290-genai-spring2026/final-project')

!pip install -q torch transformers==5.2.0 peft==0.18.1 accelerate bitsandbytes

In [ ]:
from huggingface_hub import login
login()

## Load Test Examples

Sample a few examples from the formatted dataset to use as test inputs.

In [ ]:
import json
import random

with open('data/processed/xlam_4k_formatted.jsonl') as f:
    all_examples = [json.loads(line) for line in f if line.strip()]

random.seed(42)
test_examples = random.sample(all_examples, 20)

# Separate single-call and parallel-call for analysis
single = [ex for ex in test_examples if len(ex['ground_truth']) == 1]
parallel = [ex for ex in test_examples if len(ex['ground_truth']) > 1]
print(f'Test set: {len(single)} single-call, {len(parallel)} parallel-call')

## Load Base Model (Zero-Shot)

Load Qwen2.5-3B-Instruct in 4-bit quantization for inference.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = 'left'  # left-pad for generation

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
print(f'Model loaded: {MODEL_NAME}')

## Inference Helper

Generate a function call given a user query and tool definitions.

In [ ]:
def generate_tool_call(model, tokenizer, example, max_new_tokens=512):
    """Run inference on a single example and return the generated text."""
    # Build prompt from system + user messages (exclude assistant turn)
    prompt_messages = [
        msg for msg in example['messages'] if msg['role'] != 'assistant'
    ]

    text = tokenizer.apply_chat_template(
        prompt_messages,
        tools=example['tools'],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
        )

    # Decode only the generated tokens (skip the prompt)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Score Predictions

Use the reward functions from `src/training/rewards.py` to evaluate correctness.

In [ ]:
from src.training.rewards import parse_tool_calls, correctness_reward, format_reward

def evaluate_examples(model, tokenizer, examples, label=''):
    """Run inference on examples and compute correctness + format scores."""
    completions = []
    ground_truths = []

    for i, ex in enumerate(examples):
        pred = generate_tool_call(model, tokenizer, ex)
        completions.append(pred)
        ground_truths.append(json.dumps(ex['ground_truth']))

        # Show first 3 predictions
        if i < 3:
            print(f'--- Example {i+1} ---')
            print(f'Query: {ex["messages"][1]["content"][:100]}...')
            print(f'Expected: {[a["name"] for a in ex["ground_truth"]]}')
            parsed = parse_tool_calls(pred)
            print(f'Predicted: {[c.get("name", "?") for c in parsed]}')
            print()

    correct = correctness_reward(completions, ground_truths)
    fmt = format_reward(completions)

    print(f'[{label}] Results on {len(examples)} examples:')
    print(f'  Correctness: {sum(correct)}/{len(correct)} ({100*sum(correct)/len(correct):.1f}%)')
    print(f'  Format:      {sum(1 for f in fmt if f > 0)}/{len(fmt)} valid tool_call tags')
    return correct, fmt

## Zero-Shot Baseline (Variant 1)

In [ ]:
correct_v1, fmt_v1 = evaluate_examples(model, tokenizer, test_examples, label='V1 Zero-shot')

## SFT Variant 2 (Unmasked)

Load the LoRA adapter trained on unmasked data.

In [ ]:
from peft import PeftModel

adapter_v2 = 'checkpoints/sft/sft-v2/final'
if os.path.exists(adapter_v2):
    model_v2 = PeftModel.from_pretrained(model, adapter_v2)
    model_v2.eval()
    correct_v2, fmt_v2 = evaluate_examples(model_v2, tokenizer, test_examples, label='V2 SFT')
    # Unload adapter for next variant
    model_v2 = model_v2.unload()
else:
    print(f'Adapter not found at {adapter_v2}. Run sft_training.ipynb first.')

## SFT Variant 3 (Masked)

Load the LoRA adapter trained on masked data.

In [ ]:
adapter_v3 = 'checkpoints/sft/sft-v3/final'
if os.path.exists(adapter_v3):
    model_v3 = PeftModel.from_pretrained(model, adapter_v3)
    model_v3.eval()
    correct_v3, fmt_v3 = evaluate_examples(model_v3, tokenizer, test_examples, label='V3 SFT+mask')
else:
    print(f'Adapter not found at {adapter_v3}. Run sft_training.ipynb first.')

## Summary

In [ ]:
print('Variant comparison (correctness on 20 test examples):')
print(f'  V1 Zero-shot:  {sum(correct_v1):.0f}/{len(correct_v1)}')
if 'correct_v2' in dir():
    print(f'  V2 SFT:        {sum(correct_v2):.0f}/{len(correct_v2)}')
if 'correct_v3' in dir():
    print(f'  V3 SFT+mask:   {sum(correct_v3):.0f}/{len(correct_v3)}')